### ver2의 sample분석(MBPP)_prompt_v5
프롬프트 좀 더 간결히 수정   
- execution오류 감소(Done)
- 여전히 LM디코딩 문제가 존재하긴함.   
- 프롬프트 튜닝 추가

In [1]:
import json
from collections import Counter
import statistics
import pandas as pd

# 경로를 여기서 직접 지정
path = "../../results/phase1_ver2/sample_mbpp/single_prompt_v5/details.jsonl"

def load_results(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    return data

data = load_results(path)
len(data)

df = pd.DataFrame(data)
df.head(3)

,dataset,task_id,method,model_name,attempt_idx,prompt,raw_output,generated_code,status,passed,exec_success,error_type,error_message,latency_sec,meta,timeout
0,mbpp,601,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,Write Python code only.\nSolve the problem bel...,"```python\nclass Pair:\n def __init__(self,...","class Pair:\n def __init__(self, start, end...",pass,True,True,None,None,3.315405,"{'code_exec_passed': True, 'setup_exec_passed'...",False
1,mbpp,602,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,Write Python code only.\nSolve the problem bel...,"assert first_repeated_char(""abacddbec"") == ""a""...",def first_repeated_char(s):\n # Create an e...,fail,False,True,assertion_error,"Traceback (most recent call last):\n File ""/h...",2.854972,"{'code_exec_passed': True, 'setup_exec_passed'...",False
2,mbpp,603,single,Qwen/Qwen2.5-Coder-7B-Instruct,0,Write Python code only.\nSolve the problem bel...,"assert get_ludic(15) == [1, 2, 3, 5, 7, 11, 13...",def get_ludic(n):\n # Your code here,fail,False,False,indentation_error,"Traceback (most recent call last):\n File ""/h...",1.116646,"{'code_exec_passed': False, 'setup_exec_passed...",False


In [2]:
# 전체 요약
total = len(data)
passed = sum(1 for x in data if x["status"] == "pass")
timeout = sum(1 for x in data if x["status"] == "timeout")
failed = total - passed - timeout

print("=" * 60)
print(f"📂 File: {path}")
print("📊 Overall")
print(f"Total: {total}")
print(f"Pass: {passed}")
print(f"Fail: {failed}")
print(f"Timeout: {timeout}")
print(f"Pass@1: {passed / total:.4f}")

📂 File: ../../results/phase1_ver2/sample_mbpp/single_prompt_v5/details.jsonl
📊 Overall
Total: 5
Pass: 3
Fail: 2
Timeout: 0
Pass@1: 0.6000


In [3]:
# error distribution
error_counter = Counter(
    x["error_type"] for x in data if x["error_type"] is not None
)

print("\n📉 Error Type Distribution")
for k, v in error_counter.most_common():
    print(f"{k}: {v}")
    
latencies = [x["latency_sec"] for x in data if x["latency_sec"] is not None]

if latencies:
    print("\n⏱ Latency")
    print(f"Avg: {statistics.mean(latencies):.3f}s")
    print(f"Max: {max(latencies):.3f}s")


📉 Error Type Distribution
assertion_error: 1
indentation_error: 1

⏱ Latency
Avg: 2.356s
Max: 3.315s


### 실패한 샘플의 generated_code(모델이 생성한 str) 확인

In [4]:
failures = [x for x in data if x["status"] == "fail"][:2]

for f in failures:
    print("=" * 80)
    print(f"🧪 TASK: {f['task_id']}")
    print(f"🧪 ERROR: {f['error_type']}")
    print("\n============= RAW INPUT (head) =============")
    print(f["prompt"])
    print("\n============= RAW OUTPUT (head) =============")
    print(f["raw_output"])
    print("\n============= GENERATED CODE (head) =============")
    print(f["generated_code"])
    print("\n========================================================")

🧪 TASK: 602
🧪 ERROR: assertion_error

============= RAW INPUT (head) =============
Write Python code only.
Solve the problem below.
Use the exact function name and arguments required by the test.
Include any needed helper classes or functions.

Problem:
Write a python function to find the first repeated character in a given string.

Test hint:
assert first_repeated_char("abcabc") == "a"


============= RAW OUTPUT (head) =============
assert first_repeated_char("abacddbec") == "a"
assert first_repeated_char("abcd") is None
```python
def first_repeated_char(s):
    # Create an empty dictionary to store characters and their indices
    char_index = {}
    
    # Iterate over each character in the string with its index
    for i, char in enumerate(s):
        # Check if the character is already in the dictionary
        if char in char_index:
            # If found, return the character as it's the first repeated one
            return char
        else:
            # Otherwise, add the ch

현재 이슈는 Extraneous Token Error(placeholder,,) : 공백 문자나 무의미한 문자가 나열되는 상황   
LM 디코딩 실패 이슈   

### Mode collapse
이것을 Mode collapse라고 보고자 한다.   
긴 프롬프트는 보통 설명, 코드, 다양한 자연어가 포함되는데, 모델 내부 상태가 code mode와 text mode를 왔다갔다하면서 이런 오류(공백, 무의미한 문자 나열)이 발생한 것 아닐까?   
   
관련 연구 :
- Prompt simplicity / clarity 관련 : Language Models are Few-Shot Learners(2020, openAI)


> planning-code 전략이 필요하다는 초기 근거

- Lost in the Middle: How Language Models Use Long Contexts(2023,TACL) : 이 논문은 모델이 프롬프트의 앞부분과 뒷부분은 잘 기억하지만, 중간에 있는 정보를 놓치는 현상을 다룸   
    - SLM은 어텐션 자원이 부족하여 프롬프트가 길어질수록 컨텍스트의 우선순위를 정하는 데 실패하는데, 현재 매몰되는 현상을 설명해주는 것 같음.

- textbooks Are All You Need(2023, phi-1) : Phi-1의 학습 과정을 다룬 논문으로, *'데이터의 품질'과 '코드/자연어의 분리'를 주목함.
    - 핵심 내용: 모델이 코드 데이터와 자연어 데이터를 학습할 때, 이 둘의 경계가 모호하면 추론 시 '코드 생성'과 '일반 대화' 사이에서 문법적 충돌(Interference)이 발생할 수 있음을 시사합니다. 출력물에 무의미한 문자가 섞이는 것은 모델이 다음 토큰을 예측할 때 코드의 확률 분포와 자연어의 확률 분포가 겹치면서(Overlap) 발생하는 노이즈일 가능성이 큽니다.

아래 추가실험을 했으나 오히려 성능저하

In [ ]:
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     test_hint = sample.test_list[0] if sample.test_list else ""

#     return (
#         "Write Python code only.\n"
#         "Solve the problem below.\n"
#         "Use the exact function name and arguments required by the test.\n"
#         "Include any needed helper classes or functions.\n\n"
#         f"Problem:\n{sample.problem_text}\n\n"
#         f"Test hint:\n{test_hint}\n"
#         "### Solution:\n"  # 명확한 섹션 구분
#         "```python\n"      # 모델이 바로 코드를 이어서 쓰게 함
#     )

# 📊 결과 요약
#   총 문제: 5
#   통과: 0
#   실행 성공: 0
#   pass@1: 0.0000
#   execution_success_rate: 0.0000
#   conditional_pass: 0.0000

이건 왜지???

In [ ]:
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     test_hint = sample.test_list[0] if sample.test_list else ""

#     return (
#         "<instruction>\n"
#         "Write Python code only to solve the problem. Do not provide any explanation.\n"
#         "Ensure the function name matches the test hint.\n"
#         "</instruction>\n\n"
#         f"<problem>\n{sample.problem_text}\n</problem>\n\n"
#         f"<test_hint>\n{test_hint}\n</test_hint>\n\n"
#         "<code>\n"
#     )

# 📊 결과 요약
#   총 문제: 5
#   통과: 0
#   실행 성공: 0
#   pass@1: 0.0000
#   execution_success_rate: 0.0000
#   conditional_pass: 0.0000

In [ ]:
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     test_hint = sample.test_list[0] if sample.test_list else ""

#     return (
#         "Write Python code only.\n"
#         "Solve the problem below.\n"
#         "Use the exact function name and arguments required by the test.\n"
#         "Include any needed helper classes or functions.\n\n"
#         f"Problem:\n{sample.problem_text}\n\n"
#         f"Test hint:\n{test_hint}\n"
#         f"Code:\n"
#     )

# ============================================================
# 📊 결과 요약
#   총 문제: 5
#   통과: 2
#   실행 성공: 2
#   pass@1: 0.4000
#   execution_success_rate: 0.4000
#   conditional_pass: 1.0000
# ============================================================
# 📌 MBPP Failure Breakdown
#   code_failed: 3
#   setup_failed: 0
#   test_failed: 0
#   semantic_failed: 0
#   execution_failed: 0
# ============================================================

순서는 상관없는듯

In [ ]:
# def build_mbpp_prompt(sample: MBPPSample) -> str:
#     test_hint = sample.test_list[0] if sample.test_list else ""

#     return (
#         "Write Python code only.\n"
#         "Solve the problem below.\n"
#         "Use the exact function name and arguments required by the test.\n"
#         "Include any needed helper classes or functions.\n\n"
#         f"Test hint:\n{test_hint}\n"
#         f"Problem:\n{sample.problem_text}\n\n"
#     )
    
# ============================================================
# 📊 결과 요약
#   총 문제: 5
#   통과: 3
#   실행 성공: 4
#   pass@1: 0.6000
#   execution_success_rate: 0.8000
#   conditional_pass: 0.7500
# ============================================================
# 📌 MBPP Failure Breakdown
#   code_failed: 1
#   setup_failed: 0
#   test_failed: 1
#   semantic_failed: 1
#   execution_failed: 0
# ============================================================